In [ ]:
!nvidia-smi

In [ ]:
%%writefile linear.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void linear(float *X, float *W, float *b, float *Y,
                       int batch, int in_dim, int out_dim) {
    int row = blockIdx.y*blockDim.y + threadIdx.y;
    int col = blockIdx.x*blockDim.x + threadIdx.x;
    if (row >= batch || col >= out_dim) return;
    float acc = b[col];
    for (int k = 0; k < in_dim; k++) acc += X[row*in_dim+k] * W[col*in_dim+k];
    Y[row*out_dim+col] = acc;
}

int main() {
    int B = 512, I = 1024, O = 2048;
    float *X, *W, *b, *Y;
    cudaMalloc(&X, B*I*4); cudaMalloc(&W, O*I*4);
    cudaMalloc(&b, O*4);   cudaMalloc(&Y, B*O*4);
    dim3 blk(16,16), grd((O+15)/16, (B+15)/16);
    linear<<<grd,blk>>>(X,W,b,Y,B,I,O); cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 100; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) linear<<<grd,blk>>>(X,W,b,Y,B,I,O);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    printf("Linear %dx%d->%d  %.2f TFLOPS  %.2f ms\n",
           B, I, O, 2.0*B*I*O*reps/(ms*1e9), ms/reps);
    cudaFree(X); cudaFree(W); cudaFree(b); cudaFree(Y);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o linear linear.cu && ./linear